# 📄 Module 4.1 — PDF Logical Splitting & Classification

## 1. Cài đặt

In [ ]:
!pip install pymupdf pytesseract tqdm requests -q
!apt-get install -y tesseract-ocr tesseract-ocr-vie -q 2>/dev/null

import fitz, re, json, uuid, time, pickle, hashlib, requests
import concurrent.futures
from pathlib import Path
from datetime import datetime
from tqdm.auto import tqdm
from PIL import Image
import pytesseract

print("✅ Sẵn sàng")

## 2. Cấu hình

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

BASE = '/content/drive/MyDrive/VNDigitizeComprehensiveSystem_Team03/dataset_module4'
PDF_PATH   = f'{BASE}/mix_doc/document_merged2.pdf'
OUTPUT_DIR = f'{BASE}/output'
CACHE_PATH = f'{BASE}/cache_features.pkl'

CACHE_VERSION  = "7.0"
OCR_DPI        = 200
BLANK_THRESH   = 30
TEXT_MIN_CHARS = 20
OCR_WORKERS    = 4
MIN_SEG_PAGES  = 1

RE_SO_VAN_BAN = re.compile(r'\d{1,3}/\d{4,}/[A-ZĐQĐ][\w\-]+', re.IGNORECASE)
LABEL_SET     = {"Quyết định", "Bản án dân sự", "Bản án hình sự", "Bản án hành chính"}

GT_SEGMENT_TYPES = {
     1:"Quyết định",    4:"Quyết định",    7:"Quyết định",
     9:"Bản án dân sự", 29:"Quyết định",   35:"Bản án dân sự",
    43:"Bản án hình sự",44:"Bản án hình sự",50:"Quyết định",
    52:"Quyết định",   54:"Quyết định",   57:"Quyết định",
    59:"Quyết định",   62:"Quyết định",   63:"Quyết định",
    66:"Bản án hình sự",75:"Bản án dân sự", 81:"Bản án hành chính",
   103:"Quyết định",  105:"Quyết định",  108:"Quyết định",
}

try:
    with fitz.open(PDF_PATH) as doc:
        print(f"✅ {Path(PDF_PATH).name} | {len(doc)} trang")
except Exception as e:
    print(f"⚠️  {e}")

## 3. Trích xuất văn bản

In [ ]:
def _cache_key(path):
    mtime = str(Path(path).stat().st_mtime) if Path(path).exists() else "0"
    return hashlib.md5(f"{path}|{mtime}|{CACHE_VERSION}".encode()).hexdigest()[:12]

def _load_cache(cache_path, pdf_path):
    if not Path(cache_path).exists(): return None
    try:
        b = pickle.load(open(cache_path, "rb"))
        if b.get("key") != _cache_key(pdf_path):
            print("  ⚠️  Cache lỗi thời → extract lại"); return None
        return b["features"]
    except Exception: return None

def _save_cache(cache_path, pdf_path, features):
    Path(cache_path).parent.mkdir(parents=True, exist_ok=True)
    pickle.dump({"key": _cache_key(pdf_path), "features": features}, open(cache_path, "wb"))

def _strict_header(text, n=7):
    result, first = [], ""
    for line in (text.split("\n") if text else []):
        s = line.strip()
        if not s or re.match(r'^[\d\-–—\s]{1,5}$', s): continue
        if not first: first = s
        result.append(s)
        if len(result) >= n: break
    return " ".join(result), first

def _build_feat(idx, text):
    h  = " ".join(l.strip() for l in text[:2000].split("\n")[:10] if l.strip())
    hu = h.upper()
    sh, fl = _strict_header(text)
    return {
        "page_num"      : idx + 1,
        "is_blank"      : len(text) < BLANK_THRESH,
        "header"        : h,
        "has_quoc_hieu" : "CỘNG HÒA XÃ HỘI" in hu,
        "has_toa_an"    : "TÒA ÁN NHÂN DÂN"  in hu,
        "so_vb_header"  : RE_SO_VAN_BAN.findall(h),
        "text_full"     : text,
        "strict_header" : (sh, fl),
    }

def _ocr_worker(args):
    idx, samples, w, h = args
    from PIL import Image; import pytesseract
    img = Image.frombytes("L", [w, h], samples)
    return idx, pytesseract.image_to_string(img, lang='vie', config='--oem 1 --psm 3').strip()

def batch_extract(pdf_path, cache_path=None, force=False):
    if cache_path and not force:
        cached = _load_cache(cache_path, pdf_path)
        if cached is not None:
            print(f"✅ Cache ({len(cached)} trang)"); return cached

    t0 = time.time()
    with fitz.open(pdf_path) as doc:
        raw = [(i, doc[i].get_text("text").strip()) for i in range(len(doc))]

    has_text = {i: t for i, t in raw if len(re.sub(r'\s+', '', t)) >= TEXT_MIN_CHARS}
    scan_idx = [i for i, t in raw if len(re.sub(r'\s+', '', t)) < TEXT_MIN_CHARS]
    print(f"  {len(has_text)} text-layer | {len(scan_idx)} scan")

    ocr = {}
    if scan_idx:
        pb = {}
        with fitz.open(pdf_path) as doc:
            for i in tqdm(scan_idx, desc="  render", unit="pg"):
                pix = doc[i].get_pixmap(dpi=OCR_DPI, colorspace=fitz.csGRAY)
                pb[i] = (bytes(pix.samples), pix.width, pix.height)
        args = [(i, *pb[i]) for i in scan_idx]
        with concurrent.futures.ProcessPoolExecutor(max_workers=OCR_WORKERS) as ex:
            for i, txt in tqdm(ex.map(_ocr_worker, args), total=len(args), desc="  OCR", unit="pg"):
                ocr[i] = txt

    feats = [_build_feat(i, has_text.get(i, ocr.get(i, ""))) for i in range(len(raw))]
    print(f"  ✅ {time.time()-t0:.1f}s")
    if cache_path: _save_cache(cache_path, pdf_path, feats)
    return feats

print("✅ Extract xong")

## 4. Phát hiện ranh giới

In [ ]:
_FIRST_LINE_RE = re.compile(
    r'TÒA ÁN NHÂN DÂN|VIỆN KIỂM SÁT|CỘNG HÒA XÃ HỘI'
    r'|NHÂN DANH|QUYẾT ĐỊNH\s*$|ĐÌNH CHỈ|THÔNG BÁO'
)

def _is_doc_start(strict_hdr, first_line):
    h, fl = strict_hdr.upper(), first_line.upper()
    if not _FIRST_LINE_RE.search(fl): return False, ""
    a = "CỘNG HÒA XÃ HỘI" in h or "ĐỘC LẬP" in h or "HẠNH PHÚC" in h
    b = "TÒA ÁN NHÂN DÂN"  in h or "VIỆN KIỂM SÁT" in h
    c = (bool(RE_SO_VAN_BAN.search(strict_hdr)) or "BẢN ÁN SỐ" in h
         or "NHÂN DANH" in h or bool(re.search(r'QUYẾT ĐỊNH\s*$', h, re.M))
         or "ĐÌNH CHỈ VỤ ÁN" in h or "THÔNG BÁO THỤ LÝ" in h)
    if sum([a, b, c]) >= (1 if (a and "CỘNG HÒA" in h) else 2):
        return True, "+".join(filter(None, ["quoc_hieu"*a, "co_quan"*b, "dinh_danh"*c]))
    return False, ""

def detect_boundary(feat, prev):
    if prev is None:     return True,  1.0
    if feat["is_blank"]: return False, 0.0
    if prev["is_blank"]: return True,  0.95
    is_s, sig = _is_doc_start(*feat["strict_header"])
    if is_s:
        return True, 1.0 if ("quoc_hieu" in sig and "co_quan" in sig) else 0.9
    pv = set(prev.get("so_vb_header", []))
    cv = set(feat.get("so_vb_header", []))
    if cv and pv and not cv & pv: return True, 0.85
    return False, 0.0

def get_segments(feats):
    bounds, seps = set(), set()
    for i, f in enumerate(feats):
        is_bd, _ = detect_boundary(f, feats[i-1] if i else None)
        if is_bd:         bounds.add(f["page_num"])
        if f["is_blank"]: seps.add(f["page_num"])

    segs, cur = [], []
    for f in feats:
        pn = f["page_num"]
        if f["is_blank"] or pn in seps:
            if cur: segs.append(cur); cur = []
            continue
        if pn in bounds and cur: segs.append(cur); cur = []
        cur.append(f)
    if cur: segs.append(cur)

    # merge segment 1 trang không có dấu hiệu bắt đầu rõ
    merged = [segs[0]] if segs else []
    for seg in segs[1:]:
        f0 = seg[0]
        sh = f0.get("strict_header", ("",""))[0].upper()
        clear = (f0.get("has_quoc_hieu") or f0.get("has_toa_an")
                 or bool(f0.get("so_vb_header"))
                 or "NHÂN DANH" in sh or "BẢN ÁN SỐ" in sh)
        if len(seg) <= MIN_SEG_PAGES and not clear:
            merged[-1] += seg
        else:
            merged.append(seg)
    return merged

print("✅ Boundary xong")

## 5. Phân loại

In [ ]:
_SPECIAL = {
    'QĐST-HC':"Quyết định", 'QĐPT-HC':"Quyết định",
    'QĐ-PT'  :"Quyết định", 'QĐ-ST'  :"Quyết định",
}
_KY_HIEU = [
    (re.compile(r'/QĐ[\-\w]*|/QĐST[\-\w]*|/QĐPT[\-\w]*', re.I), "Quyết định"),
    (re.compile(r'/[\w]*HC[\-]*(ST|PT|GĐT)\b',               re.I), "Bản án hành chính"),
    (re.compile(r'/[\w]*HS[\-]*(ST|PT|GĐT|SĐT)\b',           re.I), "Bản án hình sự"),
    (re.compile(r'/[\w]*(DS|HNGĐ|KDTM)[\-]*(ST|PT|GĐT)?\b',  re.I), "Bản án dân sự"),
]
_KEYWORDS = [
    ("BẢN ÁN HÌNH SỰ","Bản án hình sự",5),("BỊ CÁO","Bản án hình sự",4),
    ("VỀ TỘI","Bản án hình sự",3),        ("PHẠM TỘI","Bản án hình sự",3),
    ("BẢN ÁN DÂN SỰ","Bản án dân sự",5), ("BẢN ÁN HÔN NHÂN","Bản án dân sự",5),
    ("LY HÔN","Bản án dân sự",4),         ("TRANH CHẤP HỢP ĐỒNG","Bản án dân sự",3),
    ("NGUYÊN ĐƠN","Bản án dân sự",2),     ("BỊ ĐƠN","Bản án dân sự",2),
    ("BẢN ÁN HÀNH CHÍNH","Bản án hành chính",5),
    ("KHỞI KIỆN QUYẾT ĐỊNH HÀNH CHÍNH","Bản án hành chính",5),
    ("NGƯỜI BỊ KIỆN","Bản án hành chính",3),
]
_RE_HARD_QD = re.compile(r'QUYẾT ĐỊNH\s*(ĐÌNH CHỈ|V/V|SỐ\s*\d|XÉT XỬ|CÔNG NHẬN)', re.I)
_DINH_CHI   = ("ĐÌNH CHỈ VỤ ÁN", "ĐÌNH CHỈ XÉT XỬ", "ĐÌNH CHỈ PHIÊN TÒA")

def _by_so_vb(sv):
    su = sv.upper()
    for k, v in _SPECIAL.items():
        if k in su: return v
    for pat, lbl in _KY_HIEU:
        if pat.search(sv): return lbl
    return None

def _score(text):
    t, t5 = text[:2000].upper(), text[:500].upper()
    s = dict.fromkeys(LABEL_SET, 0)
    is_qd = bool(re.search(r'QUYẾT ĐỊNH\s*[\n\r]', t5)) or bool(
        _RE_HARD_QD.search(t5) or any(p in t5 for p in _DINH_CHI))
    if is_qd:
        s["Quyết định"] += 5
        for kw, lbl, w in _KEYWORDS:
            if lbl not in ("Bản án hình sự","Bản án dân sự") and kw in t: s[lbl] += w
    else:
        if re.search(r'QUYẾT ĐỊNH\s{0,5}(V/V|SỐ|ĐÌNH CHỈ)', t5): s["Quyết định"] += 5
        for kw, lbl, w in _KEYWORDS:
            if kw in t: s[lbl] += w
    best = max(s, key=s.get)
    if s[best] == 0: return None, 0.0
    vals = sorted(s.values(), reverse=True)
    return best, min(0.95, 0.6 + (vals[0] - vals[1]) * 0.05)

def classify_rule(pages):
    texts = [p["text_full"] for p in pages[:3]]
    h5 = texts[0][:500] if texts else ""
    if _RE_HARD_QD.search(h5.upper()) or any(p in h5.upper() for p in _DINH_CHI):
        return "Quyết định", 0.97
    for t in texts:
        for sv in RE_SO_VAN_BAN.findall(t):
            lbl = _by_so_vb(sv)
            if lbl: return lbl, 0.98
    lbl, conf = _score(" ".join(texts))
    if lbl: return lbl, conf
    if len(pages) > 1:
        for sv in RE_SO_VAN_BAN.findall(pages[1]["text_full"]):
            lbl = _by_so_vb(sv)
            if lbl: return lbl, 0.90
        lbl2, c2 = _score(pages[1]["text_full"])
        if lbl2: return lbl2, max(0.5, c2 - 0.1)
    return "Quyết định", 0.50

# Qwen verifier
QWEN_URL, QWEN_MODEL, QWEN_TO = "http://localhost:11434/api/generate", "qwen2.5:7b", 30
_P_CLASSIFY = (
    "Phân loại tài liệu pháp lý Việt Nam sau vào đúng 1 nhãn "
    "(chỉ trả nhãn, không giải thích):\n"
    "Quyết định | Bản án dân sự | Bản án hình sự | Bản án hành chính"
)

def qwen_classify(text):
    try:
        r = requests.post(QWEN_URL, timeout=QWEN_TO,
            json={"model": QWEN_MODEL, "stream": False,
                  "prompt": f"{_P_CLASSIFY}\n\n{text[:800]}\n\nNhãn:",
                  "options": {"temperature": 0}})
        raw = r.json().get("response","").strip() if r.status_code == 200 else ""
        for lbl in LABEL_SET:
            if lbl.lower() in raw.lower(): return lbl, 0.88
    except Exception: pass
    return None, 0.0

def qwen_ok():
    try: return requests.get("http://localhost:11434", timeout=3).status_code == 200
    except: return False

QWEN_ON = qwen_ok()
print(f"✅ Classifier xong | Qwen: {'online' if QWEN_ON else 'offline'}")

## 6. Pipeline

In [ ]:
_RE_NGAY = re.compile(
    r'ngày\s*(\d{1,2})\s*tháng\s*(\d{1,2})\s*năm\s*(\d{4})'
    r'|\b(\d{1,2})[/\-](\d{1,2})[/\-](\d{4})\b', re.I
)
_RE_CQ = re.compile(
    r'(TÒA ÁN[\w\s]+|UBND[\w\s,]+|BỘ[\w\s]+'
    r'|CỤC[\w\s]+|SỞ[\w\s]+|VIỆN KIỂM SÁT[\w\s]+)', re.I
)

def _extract_meta(pages, label, conf):
    head = " ".join(p["text_full"] for p in pages[:2])[:3000]
    meta = [{"field_name":"ten_loai_vb", "value":label, "confidence":round(conf,3)}]

    vb = RE_SO_VAN_BAN.findall(head)
    if vb: meta.append({"field_name":"so_van_ban", "value":vb[0].upper(), "confidence":0.97})

    m = _RE_NGAY.search(head)
    if m:
        g = m.groups()
        d, mo, y = (g[0],g[1],g[2]) if g[0] else (g[3],g[4],g[5])
        meta.append({"field_name":"ngay_ky", "value":f"{d.zfill(2)}/{mo.zfill(2)}/{y}", "confidence":0.92})

    cq = next((l.strip()[:80] for l in head.split("\n") if len(l.strip())>=5 and _RE_CQ.match(l.strip())), None)
    if not cq:
        m2 = _RE_CQ.search(head[:500])
        if m2: cq = m2.group(0)[:80].strip()
    if cq: meta.append({"field_name":"co_quan_ban_hanh", "value":cq, "confidence":0.82})

    return meta

def run_pipeline(pdf_path, output_dir, cache_path=None, gt_types=None,
                 use_qwen=True, force_extract=False):
    # Kiểm tra file
    if not Path(pdf_path).exists():
        return {"error_code": "ERR_FILE_NOT_FOUND", "error_message": f"Không tìm thấy: {pdf_path}"}
    try:
        with fitz.open(pdf_path) as doc:
            if doc.is_encrypted:
                return {"error_code": "ERR_PDF_ENCRYPTED", "error_message": "PDF có mật khẩu."}
    except Exception as e:
        return {"error_code": "ERR_FILE_CORRUPTED", "error_message": str(e)}

    t0      = time.time()
    run_dir = Path(output_dir) / datetime.now().strftime("%Y%m%d_%H%M%S")
    run_dir.mkdir(parents=True, exist_ok=True)

    # 1. Extract
    print("📑 [1/4] Extract...")
    feats = batch_extract(pdf_path, cache_path=cache_path, force=force_extract)

    # 2. Segment
    print("🔍 [2/4] Boundary...")
    segs = get_segments(feats)
    print(f"   → {len(segs)} segments")

    # 3. Classify
    print("🏷️  [3/4] Classify...")
    results = []
    for i, pages in enumerate(segs):
        rule_lbl, rule_conf = classify_rule(pages)
        qwen_lbl = None
        if use_qwen and QWEN_ON:
            qwen_lbl, _ = qwen_classify(" ".join(p["text_full"] for p in pages[:2])[:1200])
        if qwen_lbl and rule_conf < 0.80:
            lbl, conf = qwen_lbl, 0.88
        else:
            lbl, conf = rule_lbl, rule_conf

        # Phát hiện nhãn sai
        gt_lbl  = (gt_types or {}).get(pages[0]["page_num"])
        reasons = []
        if qwen_lbl and qwen_lbl in LABEL_SET and lbl != qwen_lbl:
            reasons.append(f"[{'HIGH' if conf>=0.95 else 'MED'}] rule≠qwen ({lbl}≠{qwen_lbl})")
        if conf < 0.70:  reasons.append(f"low_conf={conf:.2f}")
        if gt_lbl and gt_lbl != lbl: reasons.append(f"gt={gt_lbl}≠pred={lbl}")

        results.append({
            "segment"   : i + 1,
            "page_start": pages[0]["page_num"],
            "page_end"  : pages[-1]["page_num"],
            "type"      : lbl,
            "confidence": round(conf, 3),
            "qwen_type" : qwen_lbl,
            "mislabel"  : reasons,
            "metadata"  : _extract_meta(pages, lbl, conf),
        })

    flags = [r for r in results if r["mislabel"]]
    if flags:
        print(f"  🚨 {len(flags)} nhãn nghi sai:")
        for f in flags:
            icon = "🔴" if any("HIGH" in r for r in f["mislabel"]) else "🟡"
            print(f"    {icon} Seg {f['segment']} p{f['page_start']}-{f['page_end']}: {f['mislabel']}")
    else:
        print("  ✅ Không phát hiện nhãn sai")

    # 4. Split PDF
    print("✂️  [4/4] Split PDFs...")
    src, doc_id, sub_docs = fitz.open(pdf_path), str(uuid.uuid4())[:8], []
    for r in results:
        fname = f"sub_{r['segment']:03d}_{r['type'].replace(' ','_')}_p{r['page_start']}-{r['page_end']}.pdf"
        out   = fitz.open()
        out.insert_pdf(src, from_page=r["page_start"]-1, to_page=r["page_end"]-1)
        out.save(str(run_dir / fname)); out.close()
        sub_docs.append({"type":r["type"], "page_start":r["page_start"], "page_end":r["page_end"]})
    src.close()

    elapsed  = int((time.time()-t0)*1000)
    avg_conf = round(sum(r["confidence"] for r in results)/max(len(results),1), 3)
    types    = list({r["type"] for r in results})

    payload = {
        "document_id"       : doc_id,
        "classification"    : types[0] if len(types)==1 else "Tài liệu hỗn hợp",
        "confidence_overall": avg_conf,
        "processing_ms"     : elapsed,
        "mislabel_flags"    : len(flags),
        "metadata"          : results[0]["metadata"] if results else [],
        "sub_documents"     : sub_docs,
    }

    with open(run_dir / f"result_{doc_id}.json", "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

    print(f"\n✅ {len(sub_docs)} segs | conf={avg_conf} | {elapsed}ms")
    print(f"   {run_dir}")
    return payload

print("✅ Pipeline xong")

## 7. Chạy

In [ ]:
result = run_pipeline(
    pdf_path      = PDF_PATH,
    output_dir    = OUTPUT_DIR,
    cache_path    = CACHE_PATH,
    gt_types      = GT_SEGMENT_TYPES,
    use_qwen      = True,
    force_extract = False,  # True = bỏ cache
)
print(json.dumps(result, ensure_ascii=False, indent=2))

## 8. Đánh giá

In [ ]:
def evaluate(result, gt_types):
    if "error_code" in result:
        print(f"❌ {result['error_code']}: {result['error_message']}"); return
    subs = result.get("sub_documents", [])
    ok = total = 0
    print(f"{'Seg':>4} {'Trang':>12} {'Dự đoán':>22} {'GT':>22}  OK?")
    print("-" * 68)
    for i, s in enumerate(subs, 1):
        gt = gt_types.get(s["page_start"])
        if not gt: continue
        correct = s["type"] == gt; ok += correct; total += 1
        print(f"{i:>4} p{s['page_start']}-{s['page_end']:>3}  "
              f"{s['type']:>22}  {gt:>22}  {'✅' if correct else '❌'}")
    if total:
        print(f"\nAccuracy: {ok}/{total} = {ok/total*100:.1f}%")

if 'result' in dir():
    evaluate(result, GT_SEGMENT_TYPES)